# Cross-Modal Classification with Vast Scale Mismatch (EEG-like vs fMRI-like)

**Goal**: A classification task that can only be solved well when **cross-modal correspondences** between vastly different temporal scales are learned.

- **S1** (fine): ~128 Hz temporal resolution (e.g. 128 time points per epoch) — *EEG-like*.
- **S2** (coarse): ~1 Hz (e.g. 4–8 time points for the same epoch) — *fMRI-like*.

**Label**: Defined by a statistic that **requires both modalities aligned** (e.g. aligned correlation). So:
- **Unimodal S1**: poor — cannot compute the joint statistic.
- **Unimodal S2**: poor — too coarse; discriminative signal is in the correspondence.
- **Multimodal with correct correspondence**: much better — only then can the model use the right alignment.

We generate data, assign correspondence-dependent labels, then compare unimodal vs multimodal classifiers.

## 1. Imports and signal generator (same Kuramoto-coupled S1/S2)

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
from scipy.stats import pearsonr
from tqdm import tqdm
import matplotlib.pyplot as plt


def generate_base_signal(x, t, params):
    X, T = np.meshgrid(x, t)
    S1 = np.zeros_like(X, dtype=float)
    phases = []
    for p in params:
        component_phase = p['k'] * X - p['omega'] * T + p['phi']
        S1 += p['A'] * np.sin(component_phase)
        phases.append(component_phase)
    return S1, np.array(phases)


def kuramoto_ode(t_point, theta, t_grid, base_phases_interp, K, omega_prime):
    psi = np.array([interp_func(t_point) for interp_func in base_phases_interp])
    mean_field = np.mean(np.sin(psi - theta[:, np.newaxis]), axis=1)
    return omega_prime + K * mean_field


def generate_coupled_signal(x, t, x2, t2, s1_params, s2_params, K):
    """S1 on (x,t), S2 on (x2,t2). S2 driven by S1 at nearest x; ODE at t2."""
    S1, base_phases = generate_base_signal(x, t, s1_params)
    S2 = np.zeros((len(t2), len(x2)), dtype=float)
    omega_prime = np.array([p['omega'] for p in s2_params])
    initial_phases_s2 = np.array([p['phi'] for p in s2_params])
    amplitudes_s2 = np.array([p['A'] for p in s2_params])
    for i in range(len(x2)):
        s1_x_idx = np.argmin(np.abs(x - x2[i]))
        base_phases_at_x = base_phases[:, :, s1_x_idx]
        base_phases_interp = [
            lambda t_eval, phase_series=series: np.interp(t_eval, t, phase_series)
            for series in base_phases_at_x
        ]
        sol = solve_ivp(
            kuramoto_ode, [t[0], t[-1]], initial_phases_s2, t_eval=t2,
            args=(t, base_phases_interp, K, omega_prime), method='RK45',
        )
        S2[:, i] = np.sum(amplitudes_s2[:, np.newaxis] * np.sin(sol.y), axis=0)
    return S1, S2


## 2. Correspondence-dependent label: in-phase vs anti-phase (phase-flip)

The label must **not** be predictable from K (otherwise unimodal S2 gets high accuracy). We use a **phase-flip** design:

- **Class 1 (in-phase)**: Generate (S1, S2) normally; aligned correlation is positive.
- **Class 0 (anti-phase)**: Generate (S1, S2) then set **S2 = -S2**; aligned correlation becomes negative.

So the label is "same phase when aligned" vs "opposite phase when aligned". Unimodal S1 has no S2; unimodal S2 has symmetric marginals (flip doesn't change distribution), so neither can predict the label. Only with **correct alignment** (downsample S1 to S2 grid and compare) can we see the sign of the correlation.

In [ ]:
def downsample_to_coarse_grid(S1, c2f_t, c2f_x):
    """Average S1 over each coarse (time, space) block. S1: (n_t_fine, n_x_fine)."""
    n_t_c = len(c2f_t)
    n_x_c = len(c2f_x)
    S1_ds = np.zeros((n_t_c, n_x_c))
    for k_c in range(n_t_c):
        for i_c in range(n_x_c):
            sl_t = c2f_t[k_c]
            sl_x = c2f_x[i_c]
            S1_ds[k_c, i_c] = np.mean(S1[sl_t, sl_x])
    return S1_ds


def aligned_correlation(S1, S2, c2f_t, c2f_x):
    """Correlation between S2 and S1 downsampled to S2's grid using correspondence."""
    S1_ds = downsample_to_coarse_grid(S1, c2f_t, c2f_x)
    a, b = S1_ds.ravel(), S2.ravel()
    if np.std(a) < 1e-10 or np.std(b) < 1e-10:
        return 0.0
    r, _ = pearsonr(a, b)
    return r if not np.isnan(r) else 0.0


def build_correspondence(n_t_fine, n_x_fine, r_t, r_x):
    """Coarse grid has n_t_fine//r_t time points and n_x_fine//r_x space points."""
    n_t_c = n_t_fine // r_t
    n_x_c = n_x_fine // r_x
    c2f_t = [slice(k_c * r_t, (k_c + 1) * r_t) for k_c in range(n_t_c)]
    c2f_x = [slice(i_c * r_x, (i_c + 1) * r_x) for i_c in range(n_x_c)]
    return c2f_t, c2f_x


## 3. Generate dataset: vast scale ratio + phase-flip labels

Temporal ratio **128:4 = 32:1** (EEG-like vs fMRI-like). For each sample we draw K uniformly; 50% of samples we flip S2 → -S2 so the label is in-phase (1) vs anti-phase (0). K is varied so dynamics differ, but the label is **not** determined by K—only by whether we flipped.

In [ ]:
def generate_classification_dataset(
    num_samples,
    n_t_fine=128,
    n_x_fine=32,
    r_t=32,
    r_x=4,
    k_range=(0.5, 4.0),
    s1_params=None,
    s2_params=None,
    seed=42,
):
    """
    Generate (S1, S2) pairs with vast temporal scale ratio. Label = in-phase (1) vs
    anti-phase (0): for 50% of samples we set S2 = -S2 so aligned correlation
    is negative. Label is NOT determined by K, so unimodal S2 cannot predict it.
    """
    np.random.seed(seed)
    n_t_coarse = n_t_fine // r_t
    n_x_coarse = n_x_fine // r_x
    if s1_params is None:
        s1_params = [
            {'A': 1.0, 'k': 1.0, 'omega': 1.5, 'phi': 0},
            {'A': 0.5, 'k': 5.0, 'omega': 3.0, 'phi': np.pi / 2},
            {'A': 0.2, 'k': 15.0, 'omega': 10.0, 'phi': np.pi},
        ]
    if s2_params is None:
        s2_params = [
            {'A': 1.0, 'omega': 1.4, 'phi': 0.1},
            {'A': 0.8, 'omega': 3.2, 'phi': 0.2},
        ]

    x = np.linspace(0, 10, n_x_fine)
    t = np.linspace(0, 20, n_t_fine)
    x2 = np.linspace(0, 10, n_x_coarse)
    t2 = np.linspace(0, 20, n_t_coarse)
    c2f_t, c2f_x = build_correspondence(n_t_fine, n_x_fine, r_t, r_x)

    S1_list, S2_list, aligned_corrs, K_list, labels_list = [], [], [], [], []
    for _ in tqdm(range(num_samples), desc="Generating"):
        K = np.random.uniform(*k_range)
        S1, S2 = generate_coupled_signal(x, t, x2, t2, s1_params, s2_params, K)
        # Phase-flip: 50% anti-phase (Class 0), 50% in-phase (Class 1)
        flip = np.random.rand() < 0.5
        if flip:
            S2 = -S2
        labels_list.append(0 if flip else 1)
        corr = aligned_correlation(S1, S2, c2f_t, c2f_x)
        S1_list.append(S1)
        S2_list.append(S2)
        aligned_corrs.append(corr)
        K_list.append(K)

    aligned_corrs = np.array(aligned_corrs)
    labels = np.array(labels_list)

    return {
        'S1': np.array(S1_list),
        'S2': np.array(S2_list),
        'labels': labels,
        'aligned_corr': aligned_corrs,
        'K': np.array(K_list),
        'c2f_t': c2f_t,
        'c2f_x': c2f_x,
        'n_t_fine': n_t_fine,
        'n_x_fine': n_x_fine,
        'n_t_coarse': n_t_coarse,
        'n_x_coarse': n_x_coarse,
    }


In [ ]:
# Smaller sample counts for quick run; increase for real experiments
N_TRAIN, N_VAL, N_TEST = 400, 100, 100
data_train = generate_classification_dataset(N_TRAIN, n_t_fine=128, n_x_fine=32, r_t=32, r_x=4)
data_val   = generate_classification_dataset(N_VAL,   n_t_fine=128, n_x_fine=32, r_t=32, r_x=4, seed=43)
data_test  = generate_classification_dataset(N_TEST,  n_t_fine=128, n_x_fine=32, r_t=32, r_x=4, seed=44)

print("Train S1:", data_train['S1'].shape, "S2:", data_train['S2'].shape)
print("Temporal ratio (fine:coarse):", data_train['n_t_fine'], ":", data_train['n_t_coarse'])
print("Label balance (train):", np.bincount(data_train['labels']))


## 4. Baselines: unimodal vs multimodal

We train simple classifiers:
- **Unimodal S1**: flatten S1 or global pooling → MLP. Cannot see S2 or alignment.
- **Unimodal S2**: flatten S2 → MLP. Too coarse; label depends on aligned correlation.
- **Multimodal (naive)**: concat flattened S1 and S2 (no alignment: S2 is small, so we just concat). Without using correspondence, alignment is not explicit.
- **Multimodal (with correspondence)**: downsample S1 to S2 grid using c2f, then concat (S1_ds, S2) and classify. This uses the correct correspondence.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


def flatten_s1(S1_batch):
    """(B, T, X) -> (B, T*X)"""
    return S1_batch.reshape(S1_batch.shape[0], -1)


def flatten_s2(S2_batch):
    return S2_batch.reshape(S2_batch.shape[0], -1)


def s1_downsampled_to_s2(S1_batch, c2f_t, c2f_x):
    """(B, T_fine, X_fine) -> (B, T_coarse, X_coarse) using correspondence."""
    B = S1_batch.shape[0]
    n_t_c, n_x_c = len(c2f_t), len(c2f_x)
    out = np.zeros((B, n_t_c, n_x_c))
    for b in range(B):
        out[b] = downsample_to_coarse_grid(S1_batch[b], c2f_t, c2f_x)
    return out


def train_eval_baseline(X_train, y_train, X_val, y_val, X_test, y_test, name):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_va = scaler.transform(X_val)
    X_te = scaler.transform(X_test)
    clf = LogisticRegression(max_iter=500, random_state=0)
    clf.fit(X_tr, y_train)
    acc_val = accuracy_score(y_val, clf.predict(X_va))
    acc_test = accuracy_score(y_test, clf.predict(X_te))
    print(f"{name}: val acc = {acc_val:.3f}, test acc = {acc_test:.3f}")
    return acc_test


In [ ]:
c2f_t = data_train['c2f_t']
c2f_x = data_train['c2f_x']

X_s1_tr = flatten_s1(data_train['S1'])
X_s1_va = flatten_s1(data_val['S1'])
X_s1_te = flatten_s1(data_test['S1'])

X_s2_tr = flatten_s2(data_train['S2'])
X_s2_va = flatten_s2(data_val['S2'])
X_s2_te = flatten_s2(data_test['S2'])

S1_ds_tr = s1_downsampled_to_s2(data_train['S1'], c2f_t, c2f_x)
S1_ds_va = s1_downsampled_to_s2(data_val['S1'], c2f_t, c2f_x)
S1_ds_te = s1_downsampled_to_s2(data_test['S1'], c2f_t, c2f_x)

X_multi_naive_tr = np.hstack([X_s1_tr, X_s2_tr])
X_multi_naive_va = np.hstack([X_s1_va, X_s2_va])
X_multi_naive_te = np.hstack([X_s1_te, X_s2_te])

X_multi_aligned_tr = np.hstack([S1_ds_tr.reshape(S1_ds_tr.shape[0], -1), X_s2_tr])
X_multi_aligned_va = np.hstack([S1_ds_va.reshape(S1_ds_va.shape[0], -1), X_s2_va])
X_multi_aligned_te = np.hstack([S1_ds_te.reshape(S1_ds_te.shape[0], -1), X_s2_te])

y_tr = data_train['labels']
y_va = data_val['labels']
y_te = data_test['labels']

print("=== Baseline comparison (classification accuracy) ===")
train_eval_baseline(X_s1_tr, y_tr, X_s1_va, y_va, X_s1_te, y_te, "Unimodal S1 (fine only)")
train_eval_baseline(X_s2_tr, y_tr, X_s2_va, y_va, X_s2_te, y_te, "Unimodal S2 (coarse only)")
train_eval_baseline(X_multi_naive_tr, y_tr, X_multi_naive_va, y_va, X_multi_naive_te, y_te, "Multimodal naive (concat, no alignment)")
train_eval_baseline(X_multi_aligned_tr, y_tr, X_multi_aligned_va, y_va, X_multi_aligned_te, y_te, "Multimodal WITH correspondence (S1 downsampled to S2 grid)")


## 6. Interpretation

- **Unimodal S1**: poor — no access to S2, so cannot know in-phase vs anti-phase (~50%).
- **Unimodal S2**: poor — S2's marginal distribution is unchanged by the sign flip, so S2 alone cannot predict the label (~50%).
- **Multimodal naive**: without correct alignment (which fine block maps to which coarse point), the model cannot resolve the sign of the relationship (~50% or slightly better by chance).
- **Multimodal with correspondence**: we downsample S1 to S2's grid using the true blocks, so (S1_ds, S2) reveals the sign of correlation → high accuracy.

So classification is strong **only** when cross-modal correspondence is used. A model that must learn correspondences from (S1, S2) and labels should approach this only when alignment is learned correctly.

In [ ]:
print("\nSummary: Classification is only strong when cross-modal correspondence")
print("is used (downsample S1 to S2 grid). Unimodal and naive multimodal stay weak.")


## 5. Oracle: upper bound when correspondence is known

If we could observe the **true** aligned correlation (the same quantity that defines the label), classification would be trivial. This sets the ceiling for any method that learns the right correspondence.

In [ ]:
# Oracle: use true aligned_corr — in-phase => corr > 0, anti-phase => corr < 0
y_pred_oracle = (data_test['aligned_corr'] > 0).astype(np.int64)
acc_oracle = accuracy_score(data_test['labels'], y_pred_oracle)
print(f"Oracle (threshold aligned_corr > 0): test acc = {acc_oracle:.3f}")
print("(Multimodal-with-correspondence should approach this when alignment is correct.)")